# Time Series Model Training on Google Colab

This notebook sets up the environment and runs the PatchTST-style temporal transformer for multimodal deepfake detection.


In [ ]:
# Mount Google Drive to access your data files
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Configure paths - YOUR ACTUAL PATHS
# Path to your HDF5 file in Google Drive
HDF5_FILE_PATH = "/content/drive/MyDrive/MIT/Lab/deepfake_embeddings_2.h5"

# Path to your time_series_model.py file in Google Drive (RECOMMENDED - Option 3)
DRIVE_MODEL_PATH = "/content/drive/MyDrive/MIT/Lab/time_series_model.py"

# Path where checkpoints will be saved in Google Drive (results won't be pushed to GitHub)
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/MIT/Lab"

# Optional: Path to your repository (if you want to clone it - NOT RECOMMENDED if you don't want to push results)
REPO_URL = "https://github.com/your-username/itau-group4.git"  # Only needed for Option 1
REPO_DIR = "/content/itau-group4"

print("="*60)
print("CONFIGURED PATHS")
print("="*60)
print(f"HDF5 file path: {HDF5_FILE_PATH}")
print(f"Model file path (Drive): {DRIVE_MODEL_PATH}")
print(f"Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}")
print("="*60)


In [ ]:
# OPTION 1: Clone repository from GitHub
# ⚠️ NOT RECOMMENDED if you don't want to push results to GitHub
# Uncomment and update the repo URL if you want to use this approach

# !git clone {REPO_URL}
# %cd {REPO_DIR}
# !git checkout main  # or your branch name


In [ ]:
# OPTION 2: Upload time_series_model.py directly to Colab
# If you prefer to upload the file manually, use this cell

from google.colab import files
import os

# Create a directory for the model
!mkdir -p /content/models

# Upload the file (this will prompt you to select a file)
# uploaded = files.upload()
# 
# # Move to models directory
# for filename in uploaded.keys():
#     if filename.endswith('.py'):
#         !mv {filename} /content/models/
#         print(f"Uploaded {filename} to /content/models/")

print("If using manual upload, uncomment the code above and run it")


In [ ]:
# OPTION 3: Copy time_series_model.py from Google Drive (RECOMMENDED ✅)
# This keeps everything in Drive - no GitHub needed!
# Just upload your time_series_model.py to Google Drive and update DRIVE_MODEL_PATH above

import shutil
import os

if os.path.exists(DRIVE_MODEL_PATH):
    os.makedirs("/content/models", exist_ok=True)
    shutil.copy(DRIVE_MODEL_PATH, "/content/models/time_series_model.py")
    print(f"✓ Copied time_series_model.py from Drive to /content/models/")
    print(f"  Source: {DRIVE_MODEL_PATH}")
else:
    print(f"⚠️  File not found at {DRIVE_MODEL_PATH}")
    print("Please:")
    print("  1. Upload time_series_model.py to Google Drive")
    print("  2. Update DRIVE_MODEL_PATH in the 'setup_paths' cell above")


In [ ]:
# Install required packages
# Core dependencies for time_series_model.py
!pip install h5py numpy scikit-learn tqdm

# Install PyTorch with CUDA support (for GPU)
# This installs PyTorch 2.7.1 with CUDA 11.8 (compatible with Colab)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


In [ ]:
# Verify GPU is available
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Training will be very slow.")
    print("Make sure you've selected a GPU runtime: Runtime > Change runtime type > GPU")


In [ ]:
# Modify time_series_model.py to use correct paths and save checkpoints to Drive
# This creates a modified version that:
# 1. Uses the HDF5 path from Google Drive
# 2. Saves checkpoints to Google Drive (not local, so they persist)

import os
import re

# Find the time_series_model.py file
model_file = None
possible_paths = [
    "/content/models/time_series_model.py",  # After Option 3 copies it
    DRIVE_MODEL_PATH,  # Directly from Drive (your actual path)
    "/content/itau-group4/time_series_model.py",  # If cloned from GitHub
    "/content/time_series_model.py",  # If uploaded directly
]

for path in possible_paths:
    if os.path.exists(path):
        model_file = path
        print(f"Found model file at: {path}")
        break

if model_file:
    print(f"Found model file at: {model_file}")
    
    # Read the file
    with open(model_file, 'r') as f:
        content = f.read()
    
    # Replace the hardcoded HDF5 path
    old_hdf5_path = "/Users/jerrysheng/Desktop/Lab/deepfake_embeddings_2.h5"
    if old_hdf5_path in content:
        content = content.replace(old_hdf5_path, HDF5_FILE_PATH)
        print(f"✓ Updated HDF5 path to: {HDF5_FILE_PATH}")
    
    # Replace the checkpoint save directory
    # Look for: save_dir="./checkpoints" in train_model() call in main()
    # This pattern matches: save_dir="./checkpoints" or save_dir = "./checkpoints"
    old_checkpoint_pattern = r'save_dir\s*=\s*["\']\.\/checkpoints["\']'
    
    # Count how many matches we have
    matches = list(re.finditer(old_checkpoint_pattern, content))
    if matches:
        # Replace all occurrences
        content = re.sub(
            old_checkpoint_pattern,
            f'save_dir="{DRIVE_CHECKPOINT_DIR}"',
            content
        )
        print(f"✓ Updated {len(matches)} checkpoint directory reference(s) to: {DRIVE_CHECKPOINT_DIR}")
    else:
        print("  (No checkpoint directory found to update - may already be set to Drive path)")
    
    # Write modified version
    modified_file = "/content/models/time_series_model_modified.py"
    with open(modified_file, 'w') as f:
        f.write(content)
    
    print(f"✓ Created modified version at: {modified_file}")
    print(f"\n📁 Files will be saved to Google Drive:")
    print(f"   • Checkpoint: {DRIVE_CHECKPOINT_DIR}/best_model.pt")
    print(f"   • Training history: {DRIVE_CHECKPOINT_DIR}/training_history.json")
    print(f"   (Both files persist in Drive even after Colab session ends)")
    MODEL_FILE_TO_USE = modified_file
else:
    print("⚠️  Could not find time_series_model.py")
    print("Please make sure you've uploaded/copied the file using one of the options above")
    MODEL_FILE_TO_USE = None


In [ ]:
# Verify HDF5 file exists and check its structure
import h5py
import os

if os.path.exists(HDF5_FILE_PATH):
    print(f"✓ HDF5 file found at: {HDF5_FILE_PATH}")
    print(f"  File size: {os.path.getsize(HDF5_FILE_PATH) / 1e9:.2f} GB")
    
    # Quick check of file structure
    try:
        with h5py.File(HDF5_FILE_PATH, 'r') as f:
            print("\nFile structure:")
            def print_structure(name, obj):
                print(f"  {name}: {type(obj).__name__}")
            f.visititems(print_structure)
            
            # Check for videos group
            if 'videos' in f:
                num_videos = len(f['videos'])
                print(f"\n  Number of videos: {num_videos}")
    except Exception as e:
        print(f"  ⚠️  Error reading file: {e}")
else:
    print(f"⚠️  HDF5 file NOT found at: {HDF5_FILE_PATH}")
    print("\nPlease:")
    print("  1. Upload your deepfake_embeddings_2.h5 file to Google Drive")
    print("  2. Update HDF5_FILE_PATH in the 'setup_paths' cell above")


In [ ]:
# This cell is not needed - the script runs directly via subprocess
# Python automatically adds the script's directory to sys.path when running it
# (Removed to keep notebook clean)


In [ ]:
# Run the training script
import subprocess
import sys
import os

if MODEL_FILE_TO_USE and os.path.exists(MODEL_FILE_TO_USE):
    print("="*60)
    print("Starting Training")
    print("="*60)
    print(f"Model file: {MODEL_FILE_TO_USE}")
    print(f"HDF5 file: {HDF5_FILE_PATH}")
    print("="*60 + "\n")
    
    # Run the script
    result = subprocess.run(
        [sys.executable, MODEL_FILE_TO_USE],
        capture_output=False,
        text=True
    )
    
    if result.returncode == 0:
        print("\n✓ Training completed successfully!")
    else:
        print(f"\n⚠️  Training exited with code {result.returncode}")
else:
    print("⚠️  Cannot run training - model file not found")
    print("Please ensure time_series_model.py is available using one of the setup options above")


In [ ]:
# Locate and access your saved checkpoint (.pt file)
import os

print("="*60)
print("LOCATING YOUR CHECKPOINT FILE")
print("="*60)

# Check both Drive location and local (just in case)
checkpoint_dirs = [
    DRIVE_CHECKPOINT_DIR,  # Primary location in Drive
    "/content/models/checkpoints",  # Fallback local location
]

found_checkpoints = False
checkpoint_files = []

for checkpoint_dir in checkpoint_dirs:
    if os.path.exists(checkpoint_dir):
        print(f"\n✓ Checkpoints directory found: {checkpoint_dir}")
        files = os.listdir(checkpoint_dir)
        if files:
            print(f"\n📁 Saved checkpoints ({len(files)} files):")
            for f in files:
                filepath = os.path.join(checkpoint_dir, f)
                size = os.path.getsize(filepath) / 1e6  # MB
                print(f"  • {f} ({size:.2f} MB)")
                checkpoint_files.append((filepath, f))
            found_checkpoints = True
        else:
            print("  (no checkpoint files found yet)")
        break

if found_checkpoints and checkpoint_files:
    print("\n" + "="*60)
    print("HOW TO ACCESS YOUR CHECKPOINT:")
    print("="*60)
    
    # Extract the Drive path (remove /content/drive)
    drive_path = DRIVE_CHECKPOINT_DIR.replace("/content/drive", "")
    
    print(f"\n📍 Location in Google Drive:")
    print(f"   {drive_path}")
    print(f"\n📋 To access it:")
    print(f"   1. Go to https://drive.google.com")
    print(f"   2. Navigate to: {drive_path}")
    print(f"   3. Your .pt file(s) will be there!")
    print(f"\n💡 Tip: The file persists in Drive even after Colab session ends")
    
    # Show the actual file path
    for filepath, filename in checkpoint_files:
        if filename.endswith('.pt'):
            print(f"\n✅ Main checkpoint file: {filename}")
            print(f"   Full path: {filepath}")
else:
    print(f"\n⚠️  Checkpoints directory not found at {DRIVE_CHECKPOINT_DIR}")
    print("Training may still be in progress or checkpoints are being saved elsewhere")
    print(f"Make sure the directory exists: {DRIVE_CHECKPOINT_DIR}")


In [ ]:
# Download checkpoint to your local machine (optional)
# Your checkpoint is already in Google Drive, but you can download it here if needed

from google.colab import files
import os

checkpoint_dir = DRIVE_CHECKPOINT_DIR
if os.path.exists(checkpoint_dir):
    checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')]
    
    if checkpoint_files:
        print(f"📥 Found {len(checkpoint_files)} checkpoint file(s) to download:")
        for f in checkpoint_files:
            filepath = os.path.join(checkpoint_dir, f)
            size = os.path.getsize(filepath) / 1e6  # MB
            print(f"  • {f} ({size:.2f} MB)")
        
        print(f"\n💾 To download to your local machine:")
        print(f"   Uncomment the code below and run this cell")
        print(f"   (Or just access it from Google Drive - no download needed!)")
        
        # Uncomment below to download all checkpoint files
        # print("\n⬇️  Downloading checkpoint files...")
        # for f in checkpoint_files:
        #     filepath = os.path.join(checkpoint_dir, f)
        #     files.download(filepath)
        #     print(f"  ✓ Downloaded {f}")
        # print("\n✅ All files downloaded!")
    else:
        print("No checkpoint files found")
else:
    print(f"⚠️  Checkpoints directory not found at {checkpoint_dir}")
    print(f"   Make sure training completed successfully")


## Notes

1. **GPU Runtime**: Make sure you've selected a GPU runtime (Runtime > Change runtime type > GPU)
2. **HDF5 File**: Upload your `deepfake_embeddings_2.h5` file to Google Drive and update `HDF5_FILE_PATH`
3. **Model File**: **RECOMMENDED - Use Option 3** (copy from Drive):
   - Upload `time_series_model.py` to Google Drive
   - Update `DRIVE_MODEL_PATH` in the setup cell
   - This keeps everything in Drive - no GitHub needed!
4. **Checkpoints**: Best models are automatically saved to Google Drive at `DRIVE_CHECKPOINT_DIR`
   - They persist even after Colab session ends
   - No need to download - just access from Drive
5. **Training Time**: Training may take several hours depending on dataset size and epochs

## Setup Checklist

- [ ] Mounted Google Drive
- [ ] Updated `HDF5_FILE_PATH` to your HDF5 file location in Drive
- [ ] Updated `DRIVE_MODEL_PATH` to your `time_series_model.py` location in Drive
- [ ] Updated `DRIVE_CHECKPOINT_DIR` to where you want checkpoints saved (e.g., `/content/drive/MyDrive/itau_project/checkpoints`)
- [ ] Selected GPU runtime (Runtime > Change runtime type > GPU)
- [ ] Run Option 3 cell to copy model file from Drive

## Customization

You can modify the training configuration by editing the `ModelConfig` in the `main()` function of `time_series_model.py`, or by creating a custom training cell that imports and runs the model with different parameters.


In [ ]:
# Load and inspect the checkpoint (.pt file)
import torch
import os

print("="*60)
print("LOADING CHECKPOINT FILE")
print("="*60)

# Find the checkpoint file
checkpoint_dir = DRIVE_CHECKPOINT_DIR
checkpoint_file = None

if os.path.exists(checkpoint_dir):
    checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')]
    if checkpoint_files:
        # Use the first .pt file found (usually best_model.pt)
        checkpoint_file = os.path.join(checkpoint_dir, checkpoint_files[0])
        print(f"✓ Found checkpoint: {checkpoint_files[0]}")
    else:
        print("⚠️  No .pt files found in checkpoint directory")
else:
    print(f"⚠️  Checkpoint directory not found: {checkpoint_dir}")

if checkpoint_file and os.path.exists(checkpoint_file):
    print(f"\n📂 Loading checkpoint from:")
    print(f"   {checkpoint_file}")
    print(f"   Size: {os.path.getsize(checkpoint_file) / 1e6:.2f} MB")
    
    try:
        # Load the checkpoint
        # Note: weights_only=False is safe here since this is your own checkpoint
        # PyTorch 2.6+ defaults to weights_only=True for security, but that blocks numpy objects
        checkpoint = torch.load(checkpoint_file, map_location='cpu', weights_only=False)
        
        print("\n" + "="*60)
        print("CHECKPOINT CONTENTS:")
        print("="*60)
        
        # Show what keys are in the checkpoint
        print(f"\n📋 Keys in checkpoint:")
        for key in checkpoint.keys():
            print(f"   • {key}")
        
        # Show training info if available
        if 'epoch' in checkpoint:
            print(f"\n📊 Training Information:")
            print(f"   • Epoch: {checkpoint['epoch']}")
        
        if 'val_auroc' in checkpoint:
            print(f"   • Best Validation AUROC: {checkpoint['val_auroc']:.4f}")
        
        # Show model state dict info
        if 'model_state_dict' in checkpoint:
            model_state = checkpoint['model_state_dict']
            print(f"\n🧠 Model State Dict:")
            print(f"   • Number of parameters: {len(model_state)} layers")
            
            # Show some layer names
            print(f"\n   Sample layers (first 10):")
            for i, (name, param) in enumerate(list(model_state.items())[:10]):
                print(f"      {name}: shape {tuple(param.shape)}")
            
            if len(model_state) > 10:
                print(f"      ... and {len(model_state) - 10} more layers")
            
            # Calculate total parameters
            total_params = sum(p.numel() for p in model_state.values())
            trainable_params = sum(p.numel() for p in model_state.values() if hasattr(p, 'requires_grad') and p.requires_grad)
            print(f"\n   • Total parameters: {total_params:,}")
            print(f"   • Trainable parameters: {trainable_params:,}")
        
        # Show optimizer state if available
        if 'optimizer_state_dict' in checkpoint:
            opt_state = checkpoint['optimizer_state_dict']
            print(f"\n⚙️  Optimizer State:")
            print(f"   • Optimizer type: {type(opt_state).__name__ if hasattr(opt_state, '__class__') else 'dict'}")
            if isinstance(opt_state, dict) and 'param_groups' in opt_state:
                print(f"   • Number of parameter groups: {len(opt_state['param_groups'])}")
                if len(opt_state['param_groups']) > 0:
                    print(f"   • Learning rate: {opt_state['param_groups'][0].get('lr', 'N/A')}")
        
        print("\n" + "="*60)
        print("✅ Checkpoint loaded successfully!")
        print("="*60)
        print(f"\n💡 To use this checkpoint:")
        print(f"   1. Load it in your model: checkpoint = torch.load('{checkpoint_file}')")
        print(f"   2. Load state dict: model.load_state_dict(checkpoint['model_state_dict'])")
        print(f"   3. Optionally resume training: optimizer.load_state_dict(checkpoint['optimizer_state_dict'])")
        
    except Exception as e:
        print(f"\n❌ Error loading checkpoint: {e}")
        print(f"   The file might be corrupted or in an unexpected format")
        import traceback
        traceback.print_exc()
else:
    print(f"\n⚠️  Cannot load checkpoint - file not found")
    print(f"   Make sure training completed and checkpoint was saved to:")
    print(f"   {checkpoint_dir}")


In [ ]:
# Check for training logs or history files
import os
import json
import glob

print("="*60)
print("SEARCHING FOR TRAINING LOGS/HISTORY")
print("="*60)

# Common locations where training logs might be saved
search_locations = [
    DRIVE_CHECKPOINT_DIR,  # Same directory as checkpoint
    "/content/models",  # Model directory
    "/content",  # Root content directory
    "/content/drive/MyDrive",  # Drive root (might have logs folder)
]

# File patterns to look for
log_patterns = [
    "*.log",
    "*.txt",
    "*history*.json",
    "*metrics*.json",
    "*training*.json",
    "*results*.json",
    "*train_log*.txt",
    "*epoch*.json",
]

found_files = []

for location in search_locations:
    if os.path.exists(location):
        for pattern in log_patterns:
            files = glob.glob(os.path.join(location, "**", pattern), recursive=True)
            found_files.extend(files)

if found_files:
    print(f"\n✓ Found {len(found_files)} potential log/history file(s):\n")
    for f in found_files:
        size = os.path.getsize(f) / 1024  # KB
        print(f"  • {f} ({size:.2f} KB)")
    
    # Try to read and display JSON files (likely to contain training metrics)
    print("\n" + "="*60)
    print("ATTEMPTING TO READ TRAINING HISTORY")
    print("="*60)
    
    for f in found_files:
        if f.endswith('.json'):
            try:
                print(f"\n📄 Reading: {os.path.basename(f)}")
                with open(f, 'r') as file:
                    data = json.load(file)
                
                if isinstance(data, dict):
                    print(f"   Keys: {list(data.keys())}")
                    
                    # Look for epoch-wise metrics
                    if 'epochs' in data or 'history' in data or 'metrics' in data:
                        print(f"\n   📊 Training History Found!")
                        
                        # Try to extract epoch data
                        if 'history' in data:
                            history = data['history']
                            if isinstance(history, dict):
                                print(f"\n   Epoch-wise metrics:")
                                for metric_name, values in history.items():
                                    if isinstance(values, list):
                                        print(f"      {metric_name}: {len(values)} epochs")
                                        if len(values) > 0:
                                            print(f"         First: {values[0]:.4f}, Last: {values[-1]:.4f}")
                                        if len(values) <= 10:
                                            print(f"         All: {values}")
                        
                        # Show summary if available
                        if 'best_epoch' in data:
                            print(f"\n   Best epoch: {data['best_epoch']}")
                        if 'best_val_auroc' in data or 'best_auroc' in data:
                            best = data.get('best_val_auroc') or data.get('best_auroc')
                            print(f"   Best validation AUROC: {best:.4f}")
                
                elif isinstance(data, list):
                    print(f"   List with {len(data)} items")
                    if len(data) > 0 and isinstance(data[0], dict):
                        print(f"   Sample keys from first item: {list(data[0].keys())}")
                        if len(data) <= 5:
                            print(f"   All data: {data}")
                        else:
                            print(f"   First 3 items: {data[:3]}")
                
            except json.JSONDecodeError:
                print(f"   ⚠️  Not a valid JSON file")
            except Exception as e:
                print(f"   ⚠️  Error reading: {e}")
        
        elif f.endswith('.txt') or f.endswith('.log'):
            try:
                print(f"\n📄 Reading: {os.path.basename(f)}")
                with open(f, 'r') as file:
                    lines = file.readlines()
                
                print(f"   Total lines: {len(lines)}")
                
                # Look for epoch information
                epoch_lines = [line for line in lines if 'epoch' in line.lower() or 'auroc' in line.lower() or 'accuracy' in line.lower()]
                if epoch_lines:
                    print(f"\n   📊 Found {len(epoch_lines)} lines with training metrics:")
                    # Show last 20 metric lines (most recent)
                    for line in epoch_lines[-20:]:
                        print(f"      {line.strip()}")
                
            except Exception as e:
                print(f"   ⚠️  Error reading: {e}")

else:
    print("\n⚠️  No log or history files found")
    print("\n💡 The checkpoint only contains the best model snapshot.")
    print("   Per-epoch metrics were likely only printed to console during training.")
    print("\n   To save training history in the future, you can modify time_series_model.py to:")
    print("   1. Track metrics per epoch in a list")
    print("   2. Save them to a JSON file alongside the checkpoint")
    print("   3. Or use a logging library like TensorBoard or wandb")

print("\n" + "="*60)


In [ ]:
# Extract training metrics from checkpoint (if any additional info was saved)
# Note: The checkpoint only has the best model, but let's check if there's any other info

import torch

checkpoint_file = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pt")

if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file, map_location='cpu', weights_only=False)
    
    print("="*60)
    print("DETAILED CHECKPOINT ANALYSIS")
    print("="*60)
    
    # Show all checkpoint contents in detail
    print(f"\n📋 All checkpoint keys and their types:")
    for key, value in checkpoint.items():
        if key == 'model_state_dict':
            print(f"   • {key}: dict with {len(value)} layers")
        elif key == 'optimizer_state_dict':
            print(f"   • {key}: dict")
            if isinstance(value, dict) and 'state' in value:
                print(f"      - Contains optimizer state for {len(value['state'])} parameters")
        elif key == 'epoch':
            print(f"   • {key}: {value} (int)")
        elif key == 'val_auroc':
            print(f"   • {key}: {value:.4f} (float)")
        else:
            print(f"   • {key}: {type(value).__name__}")
            if isinstance(value, (int, float, str)):
                print(f"      Value: {value}")
    
    print(f"\n📊 Summary:")
    print(f"   • Best epoch: {checkpoint.get('epoch', 'N/A')}")
    val_auroc = checkpoint.get('val_auroc')
    if val_auroc is not None:
        print(f"   • Best validation AUROC: {val_auroc:.4f}")
    else:
        print(f"   • Best validation AUROC: N/A")
    print(f"   • Model parameters: {sum(p.numel() for p in checkpoint['model_state_dict'].values()):,}")
    
    print(f"\n💡 Note: This checkpoint only contains the best model snapshot.")
    print(f"   Per-epoch training metrics (loss, accuracy per epoch) were not saved.")
    print(f"   They were only printed to console during training.")
    print(f"\n   If you need per-epoch metrics, check:")
    print(f"   1. The Colab output/console from when training ran")
    print(f"   2. Any log files that might have been created")
    print(f"   3. Consider modifying the training code to save training history to a JSON file")
    
else:
    print("Checkpoint file not found")


In [ ]:
# Detailed breakdown of what's stored in the .pt checkpoint file
import torch
import os

checkpoint_file = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pt")

if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file, map_location='cpu', weights_only=False)
    
    print("="*60)
    print("WHAT'S STORED IN THE .PT FILE")
    print("="*60)
    
    print("\n📦 The checkpoint file contains 4 main components:\n")
    
    # 1. Epoch number
    print("1️⃣  EPOCH NUMBER")
    print("   " + "-"*50)
    print(f"   • Key: 'epoch'")
    print(f"   • Type: {type(checkpoint['epoch']).__name__}")
    print(f"   • Value: {checkpoint['epoch']}")
    print(f"   • Meaning: The epoch number when the best model was saved")
    print(f"   • Size: ~4 bytes (int)")
    
    # 2. Model state dict
    print("\n2️⃣  MODEL STATE DICT (The actual trained model)")
    print("   " + "-"*50)
    model_state = checkpoint['model_state_dict']
    print(f"   • Key: 'model_state_dict'")
    print(f"   • Type: {type(model_state).__name__}")
    print(f"   • Number of layers/parameters: {len(model_state)}")
    print(f"   • Total parameter values: {sum(p.numel() for p in model_state.values()):,}")
    print(f"   • Total size: ~{sum(p.numel() * 4 for p in model_state.values()) / 1e6:.1f} MB (assuming float32)")
    print(f"   • Contains: All the weights and biases of your trained neural network")
    print(f"   • What it includes:")
    print(f"      - Audio encoder weights (patch projection, transformer layers)")
    print(f"      - Video encoder weights (patch projection, transformer layers)")
    print(f"      - Fusion head weights (MLP layers)")
    print(f"      - Positional encodings")
    print(f"   • This is what you load into your model to use it for inference")
    
    # Show breakdown by component
    audio_params = sum(p.numel() for name, p in model_state.items() if 'audio_encoder' in name)
    video_params = sum(p.numel() for name, p in model_state.items() if 'video_encoder' in name)
    fusion_params = sum(p.numel() for name, p in model_state.items() if 'fusion_head' in name)
    
    print(f"\n   Component breakdown:")
    print(f"      - Audio encoder: {audio_params:,} parameters ({100*audio_params/sum(p.numel() for p in model_state.values()):.1f}%)")
    print(f"      - Video encoder: {video_params:,} parameters ({100*video_params/sum(p.numel() for p in model_state.values()):.1f}%)")
    print(f"      - Fusion head: {fusion_params:,} parameters ({100*fusion_params/sum(p.numel() for p in model_state.values()):.1f}%)")
    
    # 3. Optimizer state dict
    print("\n3️⃣  OPTIMIZER STATE DICT (For resuming training)")
    print("   " + "-"*50)
    opt_state = checkpoint['optimizer_state_dict']
    print(f"   • Key: 'optimizer_state_dict'")
    print(f"   • Type: {type(opt_state).__name__}")
    if isinstance(opt_state, dict):
        print(f"   • Keys: {list(opt_state.keys())}")
        if 'state' in opt_state:
            print(f"   • Number of parameter states: {len(opt_state['state'])}")
            # Calculate size
            total_opt_size = 0
            for param_state in opt_state['state'].values():
                for key, value in param_state.items():
                    if isinstance(value, torch.Tensor):
                        total_opt_size += value.numel() * 4  # float32 = 4 bytes
            print(f"   • Approximate size: ~{total_opt_size / 1e6:.1f} MB")
        if 'param_groups' in opt_state:
            print(f"   • Parameter groups: {len(opt_state['param_groups'])}")
            if len(opt_state['param_groups']) > 0:
                pg = opt_state['param_groups'][0]
                print(f"   • Learning rate: {pg.get('lr', 'N/A')}")
                print(f"   • Weight decay: {pg.get('weight_decay', 'N/A')}")
    print(f"   • Contains: Optimizer's internal state (momentum, Adam's running averages, etc.)")
    print(f"   • Purpose: Allows you to resume training from exactly where it stopped")
    print(f"   • Note: Not needed for inference, only for continuing training")
    
    # 4. Validation AUROC
    print("\n4️⃣  VALIDATION AUROC (Best performance metric)")
    print("   " + "-"*50)
    print(f"   • Key: 'val_auroc'")
    print(f"   • Type: {type(checkpoint['val_auroc']).__name__}")
    print(f"   • Value: {checkpoint['val_auroc']:.4f}")
    print(f"   • Meaning: The validation AUROC score achieved by this model")
    print(f"   • Size: ~8 bytes (float64)")
    
    # File size breakdown
    file_size = os.path.getsize(checkpoint_file)
    print("\n" + "="*60)
    print("FILE SIZE BREAKDOWN")
    print("="*60)
    print(f"\n📊 Total file size: {file_size / 1e6:.2f} MB")
    print(f"\n   Estimated breakdown:")
    print(f"   • Model weights: ~{sum(p.numel() * 4 for p in model_state.values()) / 1e6:.1f} MB (largest component)")
    print(f"   • Optimizer state: ~{total_opt_size / 1e6:.1f} MB (if calculated above)")
    print(f"   • Metadata (epoch, AUROC): <1 MB")
    print(f"   • PyTorch overhead: ~{file_size / 1e6 - sum(p.numel() * 4 for p in model_state.values()) / 1e6:.1f} MB")
    
    print("\n" + "="*60)
    print("WHAT YOU CAN DO WITH THIS CHECKPOINT")
    print("="*60)
    print("\n✅ Load the model for inference:")
    print("   model = AVTemporalModel(config)")
    print("   model.load_state_dict(checkpoint['model_state_dict'])")
    print("   model.eval()")
    print("\n✅ Resume training from this point:")
    print("   optimizer.load_state_dict(checkpoint['optimizer_state_dict'])")
    print("   start_epoch = checkpoint['epoch'] + 1")
    print("\n✅ Know the best performance:")
    print(f"   Best validation AUROC: {checkpoint['val_auroc']:.4f}")
    print(f"   Achieved at epoch: {checkpoint['epoch']}")
    
    print("\n❌ What's NOT stored:")
    print("   • Per-epoch training/validation loss")
    print("   • Per-epoch training/validation accuracy")
    print("   • Per-epoch training/validation AUROC")
    print("   • Training curves or history")
    print("   • Learning rate schedule history")
    print("   • Any other intermediate metrics")
    
    print("\n💡 To get per-epoch metrics in the future:")
    print("   Modify time_series_model.py to save a separate JSON file with training history")
    
else:
    print("Checkpoint file not found")


In [ ]:
# Understanding your model's performance: Why is AUROC 0.9955 so good?
import torch
import os

checkpoint_file = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pt")

if os.path.exists(checkpoint_file):
    checkpoint = torch.load(checkpoint_file, map_location='cpu', weights_only=False)
    val_auroc = checkpoint['val_auroc']
    
    print("="*60)
    print("UNDERSTANDING YOUR MODEL'S PERFORMANCE")
    print("="*60)
    
    print(f"\n📊 Your Validation AUROC: {val_auroc:.4f}")
    print("\n" + "="*60)
    print("WHAT DOES AUROC 0.9955 MEAN?")
    print("="*60)
    
    print("\n🎯 AUROC (Area Under ROC Curve) Interpretation:")
    print("   • Range: 0.0 to 1.0")
    print("   • 0.5 = Random guessing (worst)")
    print("   • 0.7-0.8 = Acceptable")
    print("   • 0.8-0.9 = Good")
    print("   • 0.9-0.95 = Very good")
    print("   • 0.95-0.99 = Excellent")
    print(f"   • {val_auroc:.4f} = {'🎉 Exceptional!' if val_auroc >= 0.99 else 'Excellent'}")
    
    print(f"\n📈 What this means in practice:")
    print(f"   • Your model can correctly distinguish real vs fake videos")
    print(f"     {val_auroc*100:.2f}% of the time when ranking pairs")
    print(f"   • Out of 100 random pairs (1 real, 1 fake), your model would")
    print(f"     correctly identify the real one ~{val_auroc*100:.0f} times")
    
    print("\n" + "="*60)
    print("WHY MIGHT THIS BE SO HIGH?")
    print("="*60)
    
    print("\n✅ LEGITIMATE REASONS (Good news!):")
    print("\n1. 🏗️  Strong Architecture:")
    print("   • Your PatchTST-style temporal transformer is well-suited for")
    print("     sequential deepfake detection")
    print("   • Multimodal fusion (audio + video) captures complementary signals")
    print("   • Transformer attention can learn temporal patterns effectively")
    
    print("\n2. 📚 Good Training Setup:")
    print("   • Training for 48 epochs suggests sufficient learning")
    print("   • Using validation set to select best model (prevents overfitting)")
    print("   • Model has ~7.9M parameters (good capacity without being huge)")
    
    print("\n3. 🎯 Task Characteristics:")
    print("   • Deepfake detection on temporal sequences may have clear")
    print("     distinguishing features")
    print("   • Audio-visual inconsistencies are detectable by the model")
    print("   • Your embeddings (OpenL3 audio + SENet video) may capture")
    print("     relevant signals well")
    
    print("\n4. 📊 Dataset Quality:")
    print("   • AVDeepfake1M and ShareVeo3 are diverse, high-quality datasets")
    print("   • Good balance between real and fake examples")
    print("   • Sufficient training data (100k+ samples)")
    
    print("\n" + "="*60)
    print("⚠️  POTENTIAL CONCERNS (Things to verify):")
    print("="*60)
    
    print("\n1. 🔍 Data Leakage:")
    print("   • Check: Are validation videos completely separate from training?")
    print("   • Risk: Same video appearing in both train and val sets")
    print("   • Solution: Ensure proper train/val split by video_id")
    
    print("\n2. 📉 Validation Set Too Easy:")
    print("   • Check: Is validation set representative of real-world data?")
    print("   • Risk: Validation set might be easier than test set")
    print("   • Solution: Test on held-out test set to verify generalization")
    
    print("\n3. 🎭 Overfitting to Validation:")
    print("   • Check: How many times did you select best model based on val?")
    print("   • Risk: Model might be overfitted to validation set")
    print("   • Solution: Use separate test set that was never used for selection")
    
    print("\n4. 📊 Class Imbalance:")
    print("   • Check: Is validation set balanced (50/50 real vs fake)?")
    print("   • Risk: Imbalanced sets can inflate AUROC")
    print("   • Solution: Check class distribution in validation set")
    
    print("\n5. 🔗 Dataset Similarity:")
    print("   • Check: Are train and val from same source/distribution?")
    print("   • Risk: Model might not generalize to different datasets")
    print("   • Solution: Test on completely different dataset")
    
    print("\n" + "="*60)
    print("✅ RECOMMENDED NEXT STEPS")
    print("="*60)
    
    print("\n1. 🧪 Test on Held-Out Test Set:")
    print("   • Use a test set that was NEVER used during training/validation")
    print("   • This is the true test of generalization")
    
    print("\n2. 📊 Check Other Metrics:")
    print("   • Accuracy (at optimal threshold)")
    print("   • Precision and Recall")
    print("   • F1-Score")
    print("   • Confusion matrix")
    
    print("\n3. 🎯 Test on Different Datasets:")
    print("   • Try on completely different deepfake datasets")
    print("   • Test on real-world videos (if available)")
    print("   • Check robustness to different attack types")
    
    print("\n4. 🔍 Analyze Failures:")
    print("   • Look at false positives and false negatives")
    print("   • Understand what types of videos confuse the model")
    print("   • Check if certain attack types are harder to detect")
    
    print("\n5. 📈 Compare with Baselines:")
    print("   • How does this compare to other deepfake detection methods?")
    print("   • Is 0.9955 typical for this task, or unusually high?")
    print("   • Check literature for similar architectures/datasets")
    
    print("\n" + "="*60)
    print("💡 BOTTOM LINE")
    print("="*60)
    
    print(f"\n🎉 Your model achieved AUROC {val_auroc:.4f}, which is excellent!")
    print("\n✅ This suggests:")
    print("   • The model learned meaningful patterns")
    print("   • The architecture is well-suited for the task")
    print("   • Training was successful")
    
    print("\n⚠️  But verify:")
    print("   • Test on held-out test set (not used during training)")
    print("   • Check for data leakage or validation set issues")
    print("   • Ensure generalization to different datasets")
    
    print("\n📊 If test performance matches validation:")
    print("   → You have a genuinely excellent model! 🎉")
    
    print("\n📉 If test performance is much lower:")
    print("   → Investigate overfitting or data issues")
    
else:
    print("Checkpoint file not found")


## Saving Per-Epoch Training History

The code below shows how to modify `time_series_model.py` to save per-epoch metrics to a JSON file.


In [ ]:
# Modified train_model function that saves per-epoch history
# Copy this code and replace the train_model function in time_series_model.py

import json
import os

def train_model_with_history(
    model,
    train_loader,
    val_loader,
    config,
    device,
    save_dir: str = "./checkpoints",
):
    """
    Full training loop with per-epoch history tracking.
    
    This version saves training history to a JSON file alongside the checkpoint.
    """
    os.makedirs(save_dir, exist_ok=True)
    
    # Initialize training history
    history = {
        'train_loss': [],
        'train_auroc': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_auroc': [],
        'val_accuracy': [],
        'epochs': [],
        'best_epoch': None,
        'best_val_auroc': 0.0,
        'config': {
            'learning_rate': config.learning_rate,
            'weight_decay': config.weight_decay,
            'batch_size': config.batch_size,
            'num_epochs': config.num_epochs,
            'model_dim': config.model_dim,
            'num_heads': config.num_heads,
            'num_layers': config.num_layers,
        }
    }
    
    # Loss and optimizer
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    
    best_val_auroc = 0.0
    
    for epoch in range(1, config.num_epochs + 1):
        # Train
        train_metrics = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
        print(f"Epoch {epoch} Train - Loss: {train_metrics['loss']:.4f}, AUROC: {train_metrics['auroc']:.4f}, Acc: {train_metrics.get('accuracy', 0.0):.4f}")
        
        # Store training metrics
        history['train_loss'].append(float(train_metrics['loss']))
        history['train_auroc'].append(float(train_metrics['auroc']))
        history['train_accuracy'].append(float(train_metrics.get('accuracy', 0.0)))
        history['epochs'].append(epoch)
        
        # Validate
        if val_loader is not None:
            val_metrics = validate_epoch(model, val_loader, criterion, device, epoch)
            print(f"Epoch {epoch} Val   - Loss: {val_metrics['loss']:.4f}, AUROC: {val_metrics['auroc']:.4f}, Acc: {val_metrics.get('accuracy', 0.0):.4f}")
            
            # Store validation metrics
            history['val_loss'].append(float(val_metrics['loss']))
            history['val_auroc'].append(float(val_metrics['auroc']))
            history['val_accuracy'].append(float(val_metrics.get('accuracy', 0.0)))
            
            # Save best model
            if val_metrics['auroc'] > best_val_auroc:
                best_val_auroc = val_metrics['auroc']
                history['best_epoch'] = epoch
                history['best_val_auroc'] = float(best_val_auroc)
                
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_auroc': best_val_auroc,
                }, os.path.join(save_dir, 'best_model.pt'))
                print(f"Saved best model with AUROC: {best_val_auroc:.4f}")
        
        # Save history after each epoch (so you don't lose progress if training stops)
        # This saves to the same directory as checkpoints (DRIVE_CHECKPOINT_DIR)
        history_file = os.path.join(save_dir, 'training_history.json')
        with open(history_file, 'w') as f:
            json.dump(history, f, indent=2)
    
    print(f"\n✅ Training history saved to: {history_file}")
    print(f"   (Saved in Google Drive at: {save_dir})")
    return history

print("✅ Function defined! This is a template - you'll need to modify time_series_model.py")
print("   See the next cell for instructions on how to apply this modification.")


In [ ]:
# Automatically modify time_series_model.py to add history tracking
# This will update your model file to save per-epoch metrics

import os
import re

# Find the model file
model_file = None
possible_paths = [
    "/content/models/time_series_model.py",  # After Option 3 copies it
    DRIVE_MODEL_PATH,  # Directly from Drive (your actual path)
    "/content/itau-group4/time_series_model.py",  # If cloned from GitHub
    "/content/time_series_model.py",  # If uploaded directly
]

for path in possible_paths:
    if os.path.exists(path):
        model_file = path
        print(f"Found model file: {path}")
        break

if model_file:
    print(f"Found model file: {model_file}")
    
    # Read the file
    with open(model_file, 'r') as f:
        content = f.read()
    
    # Check if json import exists
    if 'import json' not in content:
        # Add json import after other imports
        import_pattern = r'(import os\n)'
        if re.search(import_pattern, content):
            content = re.sub(import_pattern, r'\1import json\n', content)
            print("✓ Added json import")
    
    # Find the train_model function
    train_model_pattern = r'def train_model\([^)]+\):\s*"""[^"]*"""\s*os\.makedirs\(save_dir, exist_ok=True\)'
    
    if re.search(train_model_pattern, content, re.DOTALL):
        print("✓ Found train_model function")
        
        # Check if history tracking already exists
        if 'training_history.json' in content:
            print("⚠️  History tracking already exists in the file!")
            print("   Skipping modification - file already has history tracking.")
        else:
            # Add history initialization before optimizer
            history_init = '''    os.makedirs(save_dir, exist_ok=True)
    
    # Initialize training history
    history = {
        'train_loss': [],
        'train_auroc': [],
        'train_accuracy': [],
        'val_loss': [],
        'val_auroc': [],
        'val_accuracy': [],
        'epochs': [],
        'best_epoch': None,
        'best_val_auroc': 0.0,
        'config': {
            'learning_rate': config.learning_rate,
            'weight_decay': config.weight_decay,
            'batch_size': config.batch_size,
            'num_epochs': config.num_epochs,
            'model_dim': config.model_dim,
            'num_heads': config.num_heads,
            'num_layers': config.num_layers,
        }
    }
    
    # Loss and optimizer'''
            
            # Replace the os.makedirs + optimizer section
            old_section = r'os\.makedirs\(save_dir, exist_ok=True\)\s+# Loss and optimizer'
            if re.search(old_section, content):
                content = re.sub(old_section, history_init, content)
                print("✓ Added history initialization")
            
            # Add history tracking after train_metrics
            train_tracking = '''        train_metrics = train_epoch(model, train_loader, criterion, optimizer, device, epoch)
        print(f"Epoch {epoch} Train - Loss: {train_metrics['loss']:.4f}, AUROC: {train_metrics['auroc']:.4f}, Acc: {train_metrics.get('accuracy', 0.0):.4f}")
        
        # Store training metrics
        history['train_loss'].append(float(train_metrics['loss']))
        history['train_auroc'].append(float(train_metrics['auroc']))
        history['train_accuracy'].append(float(train_metrics.get('accuracy', 0.0)))
        history['epochs'].append(epoch)'''
            
            old_train_section = r'train_metrics = train_epoch\(model, train_loader, criterion, optimizer, device, epoch\)\s+print\(f"Epoch {epoch} Train'
            if re.search(old_train_section, content):
                content = re.sub(
                    r'(train_metrics = train_epoch\(model, train_loader, criterion, optimizer, device, epoch\)\s+print\(f"Epoch {epoch} Train[^"]*"\))',
                    train_tracking,
                    content
                )
                print("✓ Added training metrics tracking")
            
            # Add history tracking after val_metrics
            val_tracking = '''            val_metrics = validate_epoch(model, val_loader, criterion, device, epoch)
            print(f"Epoch {epoch} Val   - Loss: {val_metrics['loss']:.4f}, AUROC: {val_metrics['auroc']:.4f}, Acc: {val_metrics.get('accuracy', 0.0):.4f}")
            
            # Store validation metrics
            history['val_loss'].append(float(val_metrics['loss']))
            history['val_auroc'].append(float(val_metrics['auroc']))
            history['val_accuracy'].append(float(val_metrics.get('accuracy', 0.0)))'''
            
            old_val_section = r'val_metrics = validate_epoch\(model, val_loader, criterion, device, epoch\)\s+print\(f"Epoch {epoch} Val'
            if re.search(old_val_section, content):
                content = re.sub(
                    r'(val_metrics = validate_epoch\(model, val_loader, criterion, device, epoch\)\s+print\(f"Epoch {epoch} Val[^"]*"\))',
                    val_tracking,
                    content
                )
                print("✓ Added validation metrics tracking")
            
            # Update best model saving to also update history
            best_model_save = '''            if val_metrics['auroc'] > best_val_auroc:
                best_val_auroc = val_metrics['auroc']
                history['best_epoch'] = epoch
                history['best_val_auroc'] = float(best_val_auroc)
                
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_auroc': best_val_auroc,
                }, os.path.join(save_dir, 'best_model.pt'))
                print(f"Saved best model with AUROC: {best_val_auroc:.4f}")'''
            
            old_best_save = r'if val_metrics\[\'auroc\'\] > best_val_auroc:[^)]+best_val_auroc,[^)]+\)'
            if re.search(old_best_save, content, re.DOTALL):
                content = re.sub(old_best_save, best_model_save, content, flags=re.DOTALL)
                print("✓ Updated best model saving")
            
            # Add history saving at end of epoch loop
            history_save = '''                print(f"Saved best model with AUROC: {best_val_auroc:.4f}")
        
        # Save history after each epoch (so you don't lose progress if training stops)
        history_file = os.path.join(save_dir, 'training_history.json')
        with open(history_file, 'w') as f:
            json.dump(history, f, indent=2)
    
    print(f"\\n✅ Training history saved to: {history_file}")
    return history'''
            
            # Find the end of the function (before the closing of train_model)
            if 'return history' not in content:
                # Add return statement before the function ends
                content = re.sub(
                    r'(print\(f"Saved best model with AUROC: \{best_val_auroc:.4f\}"\)\s+)(\n\s*\n\s*# =)',
                    r'\1        # Save history after each epoch (so you don\'t lose progress if training stops)\n        history_file = os.path.join(save_dir, \'training_history.json\')\n        with open(history_file, \'w\') as f:\n            json.dump(history, f, indent=2)\n    \n    print(f"\\n✅ Training history saved to: {history_file}")\n    return history\2',
                    content
                )
                print("✓ Added history saving and return statement")
            
            # Write modified file
            modified_file = "/content/models/time_series_model_with_history.py"
            with open(modified_file, 'w') as f:
                f.write(content)
            
            print(f"\n✅ Created modified version with history tracking:")
            print(f"   {modified_file}")
            print(f"\n📁 Files will be saved to:")
            print(f"   • Checkpoint: {DRIVE_CHECKPOINT_DIR}/best_model.pt")
            print(f"   • Training history: {DRIVE_CHECKPOINT_DIR}/training_history.json")
            print(f"\n💡 To use it:")
            print(f"   1. Update MODEL_FILE_TO_USE in cell 8 to point to this file")
            print(f"   2. Or manually copy the changes to your original file")
            print(f"   3. Both files will be saved to your Google Drive at: {DRIVE_CHECKPOINT_DIR}")
    else:
        print("⚠️  Could not find train_model function to modify")
        print("   You may need to manually add the history tracking code")
else:
    print("⚠️  Could not find time_series_model.py")
    print("   Please make sure the file is available using one of the setup options")


In [ ]:
# Load and visualize training history (after training with history tracking)
import json
import os
import matplotlib.pyplot as plt
import numpy as np

history_file = os.path.join(DRIVE_CHECKPOINT_DIR, 'training_history.json')

if os.path.exists(history_file):
    print("="*60)
    print("LOADING TRAINING HISTORY")
    print("="*60)
    
    with open(history_file, 'r') as f:
        history = json.load(f)
    
    print(f"\n✓ Loaded training history from: {history_file}")
    print(f"\n📊 Training Summary:")
    print(f"   • Total epochs: {len(history['epochs'])}")
    print(f"   • Best epoch: {history.get('best_epoch', 'N/A')}")
    print(f"   • Best validation AUROC: {history.get('best_val_auroc', 0.0):.4f}")
    
    # Plot training curves
    epochs = history['epochs']
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('Training History', fontsize=16, fontweight='bold')
    
    # Loss
    ax1 = axes[0, 0]
    ax1.plot(epochs, history['train_loss'], label='Train Loss', marker='o', markersize=3)
    if history['val_loss']:
        ax1.plot(epochs, history['val_loss'], label='Val Loss', marker='s', markersize=3)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # AUROC
    ax2 = axes[0, 1]
    ax2.plot(epochs, history['train_auroc'], label='Train AUROC', marker='o', markersize=3)
    if history['val_auroc']:
        ax2.plot(epochs, history['val_auroc'], label='Val AUROC', marker='s', markersize=3)
        if history.get('best_epoch'):
            best_epoch = history['best_epoch']
            best_auroc = history['best_val_auroc']
            ax2.plot(best_epoch, best_auroc, 'r*', markersize=15, label=f'Best ({best_auroc:.4f})')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('AUROC')
    ax2.set_title('AUROC')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1])
    
    # Accuracy
    ax3 = axes[1, 0]
    ax3.plot(epochs, history['train_accuracy'], label='Train Accuracy', marker='o', markersize=3)
    if history['val_accuracy']:
        ax3.plot(epochs, history['val_accuracy'], label='Val Accuracy', marker='s', markersize=3)
    ax3.set_xlabel('Epoch')
    ax3.set_ylabel('Accuracy')
    ax3.set_title('Accuracy')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    ax3.set_ylim([0, 1])
    
    # Combined metrics table
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    # Create summary table
    if len(epochs) > 0:
        table_data = []
        table_data.append(['Metric', 'Train', 'Val'])
        table_data.append(['─' * 20, '─' * 15, '─' * 15])
        
        # Final values
        table_data.append(['Final Loss', f"{history['train_loss'][-1]:.4f}", 
                          f"{history['val_loss'][-1]:.4f}" if history['val_loss'] else 'N/A'])
        table_data.append(['Final AUROC', f"{history['train_auroc'][-1]:.4f}", 
                          f"{history['val_auroc'][-1]:.4f}" if history['val_auroc'] else 'N/A'])
        table_data.append(['Final Accuracy', f"{history['train_accuracy'][-1]:.4f}", 
                          f"{history['val_accuracy'][-1]:.4f}" if history['val_accuracy'] else 'N/A'])
        
        # Best values
        if history.get('best_epoch'):
            best_idx = epochs.index(history['best_epoch'])
            table_data.append(['', '', ''])
            table_data.append(['Best (Epoch {})'.format(history['best_epoch']), '', ''])
            table_data.append(['Best Val AUROC', '', f"{history['best_val_auroc']:.4f}"])
        
        table = ax4.table(cellText=table_data, cellLoc='left', loc='center',
                         colWidths=[0.4, 0.3, 0.3])
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 2)
        ax4.set_title('Summary', fontsize=12, fontweight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    # Print detailed per-epoch metrics
    print("\n" + "="*60)
    print("PER-EPOCH METRICS")
    print("="*60)
    print(f"\n{'Epoch':<8} {'Train Loss':<12} {'Train AUROC':<14} {'Train Acc':<12} {'Val Loss':<12} {'Val AUROC':<14} {'Val Acc':<12}")
    print("-" * 90)
    
    for i, epoch in enumerate(epochs[:10]):  # Show first 10
        train_loss = history['train_loss'][i]
        train_auroc = history['train_auroc'][i]
        train_acc = history['train_accuracy'][i]
        val_loss = history['val_loss'][i] if history['val_loss'] else 0.0
        val_auroc = history['val_auroc'][i] if history['val_auroc'] else 0.0
        val_acc = history['val_accuracy'][i] if history['val_accuracy'] else 0.0
        
        print(f"{epoch:<8} {train_loss:<12.4f} {train_auroc:<14.4f} {train_acc:<12.4f} {val_loss:<12.4f} {val_auroc:<14.4f} {val_acc:<12.4f}")
    
    if len(epochs) > 10:
        print("...")
        # Show last 5
        for i in range(len(epochs)-5, len(epochs)):
            epoch = epochs[i]
            train_loss = history['train_loss'][i]
            train_auroc = history['train_auroc'][i]
            train_acc = history['train_accuracy'][i]
            val_loss = history['val_loss'][i] if history['val_loss'] else 0.0
            val_auroc = history['val_auroc'][i] if history['val_auroc'] else 0.0
            val_acc = history['val_accuracy'][i] if history['val_accuracy'] else 0.0
            
            print(f"{epoch:<8} {train_loss:<12.4f} {train_auroc:<14.4f} {train_acc:<12.4f} {val_loss:<12.4f} {val_auroc:<14.4f} {val_acc:<12.4f}")
    
else:
    print("⚠️  Training history file not found")
    print(f"   Expected at: {history_file}")
    print("\n💡 To enable history tracking:")
    print("   1. Run cell 22 to automatically modify your model file")
    print("   2. Or manually add the history tracking code from cell 21")
    print("   3. Then retrain your model")


## Quick Guide: How to Generate training_history.json

**To get the JSON file with per-epoch metrics, follow these steps:**

1. **Run cell 22** - This automatically modifies your `time_series_model.py` to add history tracking
2. **Update cell 8** - Point it to use the modified file (or it will auto-detect it)
3. **Run training** - Execute cell 11 to train your model
4. **Check results** - Run cell 23 to visualize the training history

The JSON file will be saved to: `DRIVE_CHECKPOINT_DIR/training_history.json`


## Quick Access After Runtime Disconnect

**If your runtime disconnected but training completed, run these cells in order:**

1. **Cell 1** - Mount Google Drive
2. **Cell 2** - Configure paths (already set to your paths)
3. **Cell 12** - Locate checkpoint file
4. **Cell 15** - Load and inspect checkpoint
5. **Cell 23** - Load and visualize training history (if JSON exists)


In [ ]:
# QUICK ACCESS: View results after runtime disconnect
# Run this cell after re-mounting Drive and configuring paths

print("="*60)
print("QUICK ACCESS TO TRAINING RESULTS")
print("="*60)

print("\n📋 To view your results, run these cells in order:\n")

print("1️⃣  CELL 1: Mount Google Drive")
print("   → Run this first to reconnect to your Drive\n")

print("2️⃣  CELL 2: Configure paths (already set)")
print("   → Just run it to confirm paths are correct\n")

print("3️⃣  CELL 12: Locate checkpoint file")
print("   → Shows where your .pt file is saved\n")

print("4️⃣  CELL 15: Load and inspect checkpoint")
print("   → Shows checkpoint contents, model info, performance metrics\n")

print("5️⃣  CELL 23: Visualize training history")
print("   → Shows per-epoch metrics and plots (if training_history.json exists)\n")

print("="*60)
print("QUICK CHECK: Do your files exist?")
print("="*60)

import os

# Check checkpoint
checkpoint_file = os.path.join(DRIVE_CHECKPOINT_DIR, "best_model.pt")
if os.path.exists(checkpoint_file):
    size = os.path.getsize(checkpoint_file) / 1e6
    print(f"\n✅ Checkpoint found: {checkpoint_file}")
    print(f"   Size: {size:.2f} MB")
else:
    print(f"\n⚠️  Checkpoint not found at: {checkpoint_file}")

# Check training history
history_file = os.path.join(DRIVE_CHECKPOINT_DIR, "training_history.json")
if os.path.exists(history_file):
    size = os.path.getsize(history_file) / 1024
    print(f"\n✅ Training history found: {history_file}")
    print(f"   Size: {size:.2f} KB")
    print(f"   → Run CELL 23 to visualize it!")
else:
    print(f"\n⚠️  Training history not found at: {history_file}")
    print(f"   → This is normal if you trained before adding history tracking")

print("\n" + "="*60)
